# Rerankers Aren't Magic Either: When the Cross-Encoder Layer Is Worth the Cost

> 📖 **Full article on Towards Data Science:** [Rerankers Aren't Magic Either: When the Cross-Encoder Layer Is Worth the Cost](https://towardsdatascience.com/rerankers-arent-magic-either-when-the-cross-encoder-layer-is-worth-the-cost-enterprise-document-intelligence-vol-1-2bis/)
>
> This is the **runnable, code-only companion**. The explanations, the diagrams and the *why* live in the article. Here you just run the pipeline end to end. Every section below links back to the matching part of the article. **To understand any step, read the article.**


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name in ("notebooks", "book_1"):
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR
SRC_PATH = str(REPO_ROOT)
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

In [ ]:
import os
import numpy as np
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

from lib.core import get_embedding

load_dotenv()

client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
)


In [ ]:
# Side-by-side comparison: 4 embedding models and 3 cross-encoder rerankers.
# Embeddings (bi-encoders, query and passage encoded SEPARATELY):
#   - GloVe-avg : averaged 300-dim word vectors, 2014. No contextualization.
#   - MiniLM    : all-MiniLM-L6-v2, 22M params, 384-dim, 2021.
#   - ada-002   : OpenAI 2022, 1536-dim. Cheap, precomputable.
#   - 3-large   : text-embedding-3-large, OpenAI 2024, 3072-dim.
# Cross-encoder rerankers (query AND passage tokenized together, attended jointly):
#   - bge-base        : BAAI/bge-reranker-base: 278M params.
#   - bge-large       : BAAI/bge-reranker-large: 560M params.
#   - msmarco-MiniLM  : cross-encoder/ms-marco-MiniLM-L-12-v2: historical baseline.
#
# Scores live on their own scale per column (cosine in [-1, 1] for embeddings,
# logits roughly [-12, 12] for the cross-encoders). Read the RANKING inside each
# column; absolute scores across columns are NOT comparable.

from sentence_transformers import SentenceTransformer, CrossEncoder

glove = SentenceTransformer("average_word_embeddings_glove.6B.300d")
minilm = SentenceTransformer("all-MiniLM-L6-v2")

reranker_bge_base = CrossEncoder("BAAI/bge-reranker-base")
reranker_bge_large = CrossEncoder("BAAI/bge-reranker-large")
reranker_msmarco = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")

RERANKERS = {
    "bge-base": reranker_bge_base,
    "bge-large": reranker_bge_large,
    "msmarco-MiniLM": reranker_msmarco,
}

ADA   = os.getenv("MODEL_EMBED",   "text-embedding-ada-002")
LARGE = os.getenv("MODEL_EMBED_2", "text-embedding-3-large")

def _cos(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

def _oai_emb(text, model):
    return np.asarray(
        client.embeddings.create(input=text, model=model).data[0].embedding,
        dtype=np.float32,
    )

import hashlib as _hashlib
import pickle as _pickle
_CMP_CACHE = REPO_ROOT / "output" / "_book_cache" / "02_2_rerankers"
_CMP_CACHE.mkdir(parents=True, exist_ok=True)

def compare_embed_vs_rerank(query, candidates, target=None):
    """7-column comparison: 4 embeddings (cosine) + 3 cross-encoders (relevance score)."""
    if target is None:
        target = candidates[0]
    # Cache the full DataFrame keyed by (query, candidates, model names, version).
    # Every OpenAI embed + HF encode + HF rerank below is bypassed on cache hit.
    _key_mat = (
        f"{query}|||{tuple(candidates)}|||{ADA}|||{LARGE}|||glove+minilm|||"
        f"{tuple(sorted(RERANKERS))}|||v1"
    )
    _key = _hashlib.sha256(_key_mat.encode("utf-8")).hexdigest()[:16]
    _cache_path = _CMP_CACHE / f"compare_embed_vs_rerank__{_key}.pkl"
    if _cache_path.exists():
        df = _pickle.loads(_cache_path.read_bytes())
        df.attrs["target"] = target
        return df
    qg = glove.encode(query)
    qm = minilm.encode(query)
    qa = _oai_emb(query, ADA)
    ql = _oai_emb(query, LARGE)
    cv_glove  = [glove.encode(c)  for c in candidates]
    cv_minilm = [minilm.encode(c) for c in candidates]
    cv_ada    = [_oai_emb(c, ADA)   for c in candidates]
    cv_large  = [_oai_emb(c, LARGE) for c in candidates]
    pairs = [(query, c) for c in candidates]
    rerank_scores = {name: m.predict(pairs).tolist() for name, m in RERANKERS.items()}
    rows = []
    for i, c in enumerate(candidates):
        row = {
            "candidate": c,
            "GloVe-avg":  _cos(qg, cv_glove[i]),
            "MiniLM":     _cos(qm, cv_minilm[i]),
            "ada-002":    _cos(qa, cv_ada[i]),
            "3-large":    _cos(ql, cv_large[i]),
        }
        for name in RERANKERS:
            row[name] = rerank_scores[name][i]
        rows.append(row)
    df = pd.DataFrame(rows).set_index("candidate")
    df.attrs["query"] = query
    df.attrs["target"] = target
    _cache_path.write_bytes(_pickle.dumps(df))
    return df

---

📖 **Read the full article on TDS:** [Rerankers Aren't Magic Either: When the Cross-Encoder Layer Is Worth the Cost](https://towardsdatascience.com/rerankers-arent-magic-either-when-the-cross-encoder-layer-is-worth-the-cost-enterprise-document-intelligence-vol-1-2bis/)

The whole series ships on Towards Data Science: [all articles by Angela Shi](https://towardsdatascience.com/author/angela-shi/)

This notebook ships a runnable slice. **For the complete code** (every brick, the dispatcher, the schemas), get in touch on Ko-fi: https://ko-fi.com/angelashi
